![Piggy bank](piggy_bank.jpg)

Los préstamos personales son una fuente de ingresos muy rentable para los bancos. El tipo de interés típico de un préstamo a dos años en el Reino Unido es de [alrededor del 10 %](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). Puede no parecer mucho, pero solo en septiembre de 2022 los consumidores del Reino Unido pidieron prestados [unos 1.500 millones de £](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), lo que supondría aproximadamente 300 millones de £ en intereses generados por los bancos a lo largo de dos años.

Te han pedido colaborar con un banco para limpiar los datos que recopilaron en una campaña de marketing reciente, cuyo objetivo era que los clientes contrataran un préstamo personal. Planean realizar más campañas en el futuro, así que quieren que te asegures de que los datos se ajustan a la estructura y a los tipos de datos específicos que indican. De este modo, podrán usar los datos limpios que entregues para configurar una base de datos en PostgreSQL que almacene la información de esta campaña y facilite la importación de datos de campañas futuras.

Te han proporcionado un archivo csv llamado `"bank_marketing.csv"`, que tendrás que limpiar, reformatear y dividir, guardando tres archivos csv finales. En concreto, los tres archivos deben tener los nombres y contenidos que se detallan a continuación:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | ID del cliente | N/A |
| `age` | `integer` | Edad del cliente en años | N/A |
| `job` | `object` | Tipo de empleo del cliente | Cambia `"."` por `"_"` |
| `marital` | `object` | Estado civil del cliente | N/A |
| `education` | `object` | Nivel educativo del cliente | Cambia `"."` por `"_"` y `"unknown"` por `np.NaN` |
| `credit_default` | `bool` | Si el crédito del cliente está en situación de impago | Convierte al tipo de datos `boolean`:<br> `1` si `"yes"`, en caso contrario `0` |
| `mortgage` | `bool` | Si el cliente tiene una hipoteca (préstamo de vivienda) | Convierte al tipo de datos booleano:<br> `1` si `"yes"`, en caso contrario `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | ID del cliente | N/A |
| `number_contacts` | `integer` | Número de intentos de contacto con el cliente en la campaña actual | N/A |
| `contact_duration` | `integer` | Duración del último contacto en segundos | N/A |
| `previous_campaign_contacts` | `integer` | Número de intentos de contacto con el cliente en la campaña anterior | N/A |
| `previous_outcome` | `bool` | Resultado de la campaña anterior | Convierte al tipo de datos booleano:<br> `1` si `"success"`, en caso contrario `0`. |
| `campaign_outcome` | `bool` | Resultado de la campaña actual | Convierte al tipo de datos booleano:<br> `1` si `"yes"`, en caso contrario `0`. |
| `last_contact_date` | `datetime` | Última fecha en la que se contactó con el cliente | Crea a partir de la combinación de `day`, `month` y una columna `year` nueva (que debe tener el valor `2022`); <br> **Formato =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | ID del cliente | N/A |
| `cons_price_idx` | `float` | Índice de precios al consumidor (indicador mensual) | N/A |
| `euribor_three_months` | `float` | Tipo euribor a tres meses (Euro Interbank Offered Rate, indicador diario) | N/A |

In [61]:
import pandas as pd
import numpy as np

# Start coding here...

In [62]:
df = pd.read_csv("bank_marketing.csv")

for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]:
    print(col)
    print("--------------")
    print(df[col].value_counts())

credit_default
--------------
no         32588
unknown     8597
yes            3
Name: credit_default, dtype: int64
mortgage
--------------
yes        21576
no         18622
unknown      990
Name: mortgage, dtype: int64
previous_outcome
--------------
nonexistent    35563
failure         4252
success         1373
Name: previous_outcome, dtype: int64
campaign_outcome
--------------
no     36548
yes     4640
Name: campaign_outcome, dtype: int64


In [63]:
df.tail(20)

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
41168,41168,38,entrepreneur,married,university.degree,no,no,nov,5,144,2,0,nonexistent,94.767,1.030,no
41169,41169,62,services,married,high.school,no,yes,nov,21,154,5,0,nonexistent,94.767,1.030,no
41170,41170,40,management,divorced,university.degree,no,yes,nov,21,293,2,4,failure,94.767,1.030,no
41171,41171,33,student,married,professional.course,no,yes,nov,10,112,1,0,nonexistent,94.767,1.031,yes
41172,41172,31,admin.,single,university.degree,no,yes,nov,6,353,1,0,nonexistent,94.767,1.031,yes
41173,41173,62,retired,married,university.degree,no,yes,nov,6,329,1,2,failure,94.767,1.031,yes
41174,41174,62,retired,married,university.degree,no,yes,nov,14,208,1,6,success,94.767,1.031,yes
41175,41175,34,student,single,unknown,no,yes,nov,24,180,1,2,failure,94.767,1.031,no
41176,41176,38,housemaid,divorced,high.school,no,yes,nov,6,360,1,0,nonexistent,94.767,1.031,no
41177,41177,57,retired,married,professional.course,no,yes,nov,25,124,6,0,nonexistent,94.767,1.031,no


In [64]:
dfa=df.copy()
dfa.info()
dfa.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   client_id                   41188 non-null  int64  
 1   age                         41188 non-null  int64  
 2   job                         41188 non-null  object 
 3   marital                     41188 non-null  object 
 4   education                   41188 non-null  object 
 5   credit_default              41188 non-null  object 
 6   mortgage                    41188 non-null  object 
 7   month                       41188 non-null  object 
 8   day                         41188 non-null  int64  
 9   contact_duration            41188 non-null  int64  
 10  number_contacts             41188 non-null  int64  
 11  previous_campaign_contacts  41188 non-null  int64  
 12  previous_outcome            41188 non-null  object 
 13  cons_price_idx              411

client_id                     0
age                           0
job                           0
marital                       0
education                     0
credit_default                0
mortgage                      0
month                         0
day                           0
contact_duration              0
number_contacts               0
previous_campaign_contacts    0
previous_outcome              0
cons_price_idx                0
euribor_three_months          0
campaign_outcome              0
dtype: int64

In [65]:
#Correcciones en los datos
dfa['job']=dfa['job'].str.replace('.','_')
dfa['education']=dfa['education'].str.replace('.','_')
dfa['education']=dfa['education'].replace('unknown',np.nan)
def booleanos(x):
    if x=='yes':
        return 1
    else:
        return 0
dfa['flag']=dfa['credit_default'].apply(booleanos).astype(bool)      
dfa['flag2']=dfa['mortgage'].apply(booleanos).astype(bool)     



In [66]:
client=dfa[['client_id','age','job','marital','education','flag','flag2']]
client.rename(columns={'flag': 'credit_default', 'flag2': 'mortgage'}, inplace=True)
client.to_csv('client.csv', index=False)

In [67]:
dfa.head()

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome,flag,flag2
0,0,56,housemaid,married,basic_4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no,False,False
1,1,57,services,married,high_school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no,False,False
2,2,37,services,married,high_school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no,False,True
3,3,40,admin_,married,basic_6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no,False,False
4,4,56,services,married,high_school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no,False,False


In [68]:
def booleanos2(x):
    if x=='success':
        return 1
    else:
        return 0
dfa['flag3']=dfa['previous_outcome'].apply(booleanos2).astype(bool)  
dfa['flag4']=dfa['campaign_outcome'].apply(booleanos).astype(bool)  
dfa['year']=2022
dfa['last_contact_date']= pd.to_datetime(dfa['day'].astype(str) + ' ' +
    dfa['month'] + ' ' +
    dfa['year'].astype(str)
)
campaign=dfa[['client_id','number_contacts','contact_duration','previous_campaign_contacts','flag3','flag4','last_contact_date']]
campaign.rename(columns={'flag3': 'previous_outcome', 'flag4': 'campaign_outcome'}, inplace=True)
campaign['last_contact_date']=campaign['last_contact_date'].dt.strftime('%Y-%m-%d')
campaign.head()
campaign.to_csv('campaign.csv', index=False)

In [69]:
economics=dfa[['client_id','cons_price_idx','euribor_three_months']]
economics.head()
economics.to_csv('economics.csv', index=False)